# Representing Text: from Documents to Vectors

We turn a real corpus into **vectors**, see what the vector space is good for, weight it with
TF-IDF, and finish by adding features that are not words at all.

The corpus is **20 newsgroups**, about 18,000 Usenet posts from 1993 in 20 categories. We take
four categories to keep everything fast.

In [1]:
# Run once if you are not using the course environment.
# %pip install -q numpy pandas scipy scikit-learn matplotlib spacy
# !python -m spacy download en_core_web_sm

In [2]:
from sklearn.datasets import fetch_20newsgroups

categories = ["comp.graphics", "rec.autos", "sci.space", "talk.politics.mideast"]

# NOTE: no remove=(...) here. We want the posts exactly as they were written,
# headers, signatures, quoted replies and all.
train = fetch_20newsgroups(subset="train", categories=categories)
test = fetch_20newsgroups(subset="test", categories=categories)

x_train, y_train = train.data, train.target
x_test, y_test = test.data, test.target

print(len(x_train), "training documents,", len(x_test), "test documents")
print("categories:", train.target_names)

2335 training documents, 1555 test documents
categories: ['comp.graphics', 'rec.autos', 'sci.space', 'talk.politics.mideast']


In [3]:
print(x_train[3][:1200])

From: MLINDROOS@FINABO.ABO.FI (Marcus Lindroos INF)
Subject: Into Infinity?(WAS:Re: *Doppelganger* (was Re: Vulcan?)
In-Reply-To: sysmgr@king.eng.umd.edu's message of 15 Apr 1993 22:22:19 GMT
Organization: Abo Akademi University, Finland
X-News-Reader: VMS NEWS 1.24
Lines: 36

In <1qkn6rINNett@mojo.eng.umd.edu> sysmgr@king.eng.umd.edu writes:

> In article <1993Apr15.170048.1@fnalf.fnal.gov>, higgins@fnalf.fnal.gov (Bill Higgins-- Beam Jockey) writes:
> 
> >This was known as *Journey to the Far Side of the Sun* in the United
> >States and as *Doppelganger* in the U.K... Later, they went
> >on to do more live-action SF series: *UFO* and *Space: 1999*.
> >
> >The astronomy was lousy, but the lifting-body spacecraft, VTOL
> >airliners, and mighty Portugese launch complex were *wonderful* to
> >look at.

Exactly. Some of the SPACE:1999 effects remain first-rate even today. 
 
> They recycled a lot of models and theme music for UFO.  Some of the
> concepts even showed up in SPACE: 1999. 
> 

Everything above the empty line is a **header**. Lines starting with `>` are the message
being replied to. Many posts end with a **signature**. None of it is the language we want to model,
and all of it will end up in the vocabulary if we do nothing.

How common is each kind of noise?

In [4]:
import re

def share(pattern, docs=x_train):
    """Percentage of documents matching a regex."""
    return 100 * sum(bool(re.search(pattern, d, re.M)) for d in docs) / len(docs)

print(f"email address      {share(r'[\w.\-]+@[\w.\-]+'):5.1f}%")
print(f"quoted reply (^>)  {share(r'^>'):5.1f}%")
print(f"'writes:' line     {share(r'writes:'):5.1f}%")
print(f"signature '-- '    {share(r'^-- *$'):5.1f}%")
print(f"ALL-CAPS word      {share(r'\b[A-Z]{4,}\b'):5.1f}%")

email address       99.7%
quoted reply (^>)   49.1%
'writes:' line      51.8%
signature '-- '     31.1%
ALL-CAPS word       74.0%


## Tokenization

`CountVectorizer` does tokenization, lowercasing and counting in one object.
Try re-running the next cells without the `min_df` parameter and watch the vocabulary size.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

In [6]:
vect = CountVectorizer(min_df=5)  # tokenization and frequency count

print('fit')
vect.fit(x_train)
print('transform')
X_train_tok = vect.transform(x_train)
print('done')

# the two steps above can be condensed in a single step that processes train
# data only once.

# print('fit_transform')
# X_train_tok = vect.fit_transform(x_train)
# print('done')

X_test_tok =vect.transform(x_test)

fit
transform
done


In [7]:
len(vect.vocabulary_)

9468

In [8]:
vect.get_feature_names_out()[:50]   # the first 50 features, alphabetically

array(['00', '000', '0005', '01', '01wb', '02', '02238', '024246', '03',
       '04', '0400', '05', '0565', '06', '07', '08', '09', '0l', '10',
       '100', '1000', '100m', '101', '102', '1024x768', '1026', '103',
       '104', '1045', '105', '106', '107', '10792', '109', '10k', '10w40',
       '11', '110', '1100', '111', '112', '113', '114', '115', '1150',
       '11632', '117', '12', '120', '1200'], dtype=object)

In [9]:
print(X_train_tok[0,:])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 178 stored elements and shape (1, 9468)>
  Coords	Values
  (0, 36)	1
  (0, 42)	1
  (0, 182)	1
  (0, 192)	1
  (0, 200)	1
  (0, 209)	1
  (0, 349)	1
  (0, 788)	1
  (0, 812)	1
  (0, 850)	2
  (0, 913)	1
  (0, 926)	5
  (0, 972)	1
  (0, 978)	1
  (0, 1090)	2
  (0, 1099)	1
  (0, 1217)	1
  (0, 1249)	1
  (0, 1275)	1
  (0, 1277)	1
  (0, 1353)	1
  (0, 1388)	4
  (0, 1424)	1
  (0, 1441)	1
  (0, 1447)	1
  :	:
  (0, 8810)	2
  (0, 8937)	1
  (0, 8938)	1
  (0, 8949)	1
  (0, 8954)	1
  (0, 9019)	1
  (0, 9160)	1
  (0, 9161)	1
  (0, 9171)	1
  (0, 9172)	1
  (0, 9184)	1
  (0, 9242)	1
  (0, 9259)	1
  (0, 9260)	1
  (0, 9269)	1
  (0, 9271)	1
  (0, 9291)	1
  (0, 9294)	1
  (0, 9308)	5
  (0, 9337)	1
  (0, 9360)	1
  (0, 9378)	2
  (0, 9434)	4
  (0, 9440)	2
  (0, 9442)	1


Some scikit-learn modules implement an inverse_transform method to reconstruct input from their output.
Let's print out the feature names and their frequency for a document. Note that frequency info is lost.

In [10]:
vect.inverse_transform(X_train_tok[0,:])

[array(['11', '114', '1975', '1983', '1991', '1993apr15', '34', 'agree',
        'air', 'all', 'an', 'and', 'any', 'anyway', 'article', 'as',
        'auto', 'away', 'baby', 'back', 'be', 'behanna', 'best', 'big',
        'bike', 'bolt', 'bottom', 'buy', 'by', 'ca', 'can', 'cars',
        'chris', 'clean', 'cleaner', 'cleaners', 'cleaning', 'clutch',
        'coat', 'coffee', 'com', 'concerning', 'dan', 'day', 'dcd',
        'disclaimer', 'do', 'dod', 'doesn', 'dogs', 'done', 'doug', 'dry',
        'eaten', 'engine', 'etc', 'excess', 'experienced', 'find', 'fixes',
        'follow', 'for', 'friends', 'from', 'fun', 'geoquest', 'get',
        'getting', 'go', 'handle', 'hands', 'hang', 'hard', 'height',
        'hence', 'high', 'holes', 'hose', 'houston', 'how', 'if', 'in',
        'inc', 'instructions', 'into', 'is', 'it', 'just', 'kit',
        'laboratory', 'lady', 'later', 'lines', 'little', 'lots', 'make',
        'man', 'mechanic', 'metal', 'method', 'miscellaneous', 'my',
       

Let's attach frequency data to features

In [11]:
for feat,freq in zip(vect.inverse_transform(X_train_tok[0,:])[0], X_train_tok[0,:].data):
  print(feat,freq)

11 1
114 1
1975 1
1983 1
1991 1
1993apr15 1
34 1
agree 1
air 1
all 2
an 1
and 5
any 1
anyway 1
article 2
as 1
auto 1
away 1
baby 1
back 1
be 1
behanna 4
best 1
big 1
bike 1
bolt 1
bottom 1
buy 3
by 2
ca 2
can 6
cars 1
chris 2
clean 1
cleaner 2
cleaners 1
cleaning 1
clutch 1
coat 1
coffee 2
com 4
concerning 1
dan 1
day 1
dcd 1
disclaimer 1
do 2
dod 1
doesn 2
dogs 1
done 1
doug 1
dry 1
eaten 1
engine 3
etc 1
excess 1
experienced 1
find 1
fixes 1
follow 1
for 2
friends 1
from 3
fun 1
geoquest 1
get 1
getting 1
go 1
handle 1
hands 2
hang 1
hard 1
height 1
hence 1
high 1
holes 1
hose 1
houston 1
how 1
if 3
in 4
inc 1
instructions 1
into 2
is 3
it 4
just 1
kit 2
laboratory 1
lady 1
later 1
lines 1
little 4
lots 1
make 2
man 1
mechanic 1
metal 1
method 1
miscellaneous 1
my 2
name 1
nec 4
needs 1
nj 2
not 2
now 1
of 6
off 4
on 3
one 1
organization 1
pack 1
paint 1
paper 1
part 1
parts 3
performance 1
pretty 1
put 1
raised 1
re 3
rebuild 4
red 1
remember 1
right 2
rule 1
school 1
se 1
slb 2
slc

### Preprocessing is a set of decisions

Every option below is one of the choices from the text processing lecture, and each one
changes the vocabulary. There is no single right answer, but there are consequences.

In [12]:
configs = {
    "default (lowercase only)": dict(),
    "+ english stopwords": dict(stop_words="english"),
    "+ min_df=2": dict(stop_words="english", min_df=2),
    "+ min_df=5, max_df=0.5": dict(stop_words="english", min_df=5, max_df=0.5),
    "letters only, 3+ chars": dict(stop_words="english", min_df=2,
                                   token_pattern=r"(?u)\b[a-zA-Z]{3,}\b"),
}

for name, kw in configs.items():
    X = CountVectorizer(**kw).fit_transform(x_train)
    print(f"{name:28s} vocabulary {X.shape[1]:6d}")

default (lowercase only)     vocabulary  38130
+ english stopwords          vocabulary  37827
+ min_df=2                   vocabulary  20261
+ min_df=5, max_df=0.5       vocabulary   9170
letters only, 3+ chars       vocabulary  17581


Dropping words that occur in a single document (`min_df=2`) halves the vocabulary.
That is Zipf's law: most types are hapax legomena.

### Using a different tokenizer

`CountVectorizer` has its own tokenizer: lowercase, then keep sequences of two or more word
characters (`token_pattern`). Two hooks let you replace it:

- **`tokenizer=`** replaces only the step that splits a string into tokens. Lowercasing and
  n-gram generation still happen around it.
- **`analyzer=`** replaces the whole chain from raw string to list of tokens: preprocessing,
  tokenizing and n-grams.

Either one makes `token_pattern` irrelevant, which is what the warning is about.

In [13]:
# %pip install -q nltk
# import nltk; nltk.download("punkt_tab")

import nltk
from nltk.stem.porter import PorterStemmer

sample = "The cars were running faster than the other car's engines in 2026."

default_analyzer = CountVectorizer().build_analyzer()    # what sklearn does by default

print("sklearn default :", default_analyzer(sample))
print("nltk word_tokenize:", nltk.word_tokenize(sample))

sklearn default : ['the', 'cars', 'were', 'running', 'faster', 'than', 'the', 'other', 'car', 'engines', 'in', '2026']
nltk word_tokenize: ['The', 'cars', 'were', 'running', 'faster', 'than', 'the', 'other', 'car', "'s", 'engines', 'in', '2026', '.']


NLTK keeps the punctuation and splits the clitic (`car` + `'s`), scikit-learn throws both
away. Neither is right in general: they are different decisions about what a token is.

An **analyzer** is just a function from a string to a list of tokens, so we can put a stemmer
inside it:

In [14]:
stemmer = PorterStemmer()

def stem_analyzer(doc):
    """Lowercase and split as sklearn does, then stem each token."""
    return [stemmer.stem(t) for t in default_analyzer(doc)]

print("default :", default_analyzer(sample))
print("stemmed :", stem_analyzer(sample))

default : ['the', 'cars', 'were', 'running', 'faster', 'than', 'the', 'other', 'car', 'engines', 'in', '2026']
stemmed : ['the', 'car', 'were', 'run', 'faster', 'than', 'the', 'other', 'car', 'engin', 'in', '2026']


Stemming collapses inflected forms, so the
vocabulary shrinks:

In [15]:
vect_stem = CountVectorizer(analyzer=stem_analyzer, min_df=5)
X_stem = vect_stem.fit_transform(x_train)

print(f"default vectorizer  {X_train_tok.shape[1]:6d} features")
print(f"stemming analyzer   {X_stem.shape[1]:6d} features")

for w in ["car", "engine", "run"]:
    variants = [t for t in vect.get_feature_names_out() if t.startswith(w)]
    print(f"\n{w!r} in the default vocabulary: {variants[:8]}")
    print(f"{w!r} after stemming:           {[t for t in vect_stem.get_feature_names_out() if t.startswith(w)][:8]}")

default vectorizer    9468 features
stemming analyzer     7372 features

'car' in the default vocabulary: ['car', 'caratzas', 'carbon', 'carbs', 'card', 'carderock', 'cards', 'care']
'car' after stemming:           ['car', 'caratza', 'carb', 'carbon', 'card', 'carderock', 'care', 'career']

'engine' in the default vocabulary: ['engine', 'engineer', 'engineering', 'engineers', 'engines']
'engine' after stemming:           []

'run' in the default vocabulary: ['run', 'running', 'runs']
'run' after stemming:           ['run']


Note the empty list for `engine`: the Porter stemmer maps *engine*, *engineer*, *engines* to **`engin`**, which is not a word. That is the trade-off from the text processing lecture: a stemmer merges inflected forms, but its output need not be a real word, so the features become harder to read.

A tokenizer written this way is **slower** and it has
to be available at prediction time too, since the same analyzer must run on new documents.

### Sparsity

The document-term matrix is almost all zeros. That is why scikit-learn keeps it in a
**sparse** format and never builds the dense version.

In [16]:
X_train_tok = vect.transform(x_train)

n_cells = X_train_tok.shape[0] * X_train_tok.shape[1]
print("matrix shape      ", X_train_tok.shape)
print(f"non-zero cells     {X_train_tok.nnz} ({100 * X_train_tok.nnz / n_cells:.2f}%)")
print(f"sparse, in memory  {X_train_tok.data.nbytes / 1e6:.1f} MB")
print(f"dense, would be    {n_cells * 8 / 1e6:.0f} MB")

matrix shape       (2335, 9468)
non-zero cells     335638 (1.52%)
sparse, in memory  2.7 MB
dense, would be    177 MB


## Weighting

`TfidfTransformer` takes the counts and reweights them. Note the **contract**: `fit` on the
training set only, then `transform` both. The IDF values must come from the training corpus,
otherwise the test set has leaked into the representation.

In [17]:
tfidf = TfidfTransformer()  # weighting
tfidf.fit(X_train_tok)
X_train_vec = tfidf.transform(X_train_tok)
X_test_vec = tfidf.transform(X_test_tok)

In [18]:
X_test_tok = vect.transform(x_test)
X_test_vec = tfidf.transform(X_test_tok)
print(X_train_vec.shape, X_test_vec.shape)

(2335, 9468) (1555, 9468)


The same document, now as weights instead of counts. Compare the two columns:

In [19]:
doc_id = 3
for feat, weight, freq in zip(vect.inverse_transform(X_train_vec[doc_id, :])[0],
                              X_train_vec[doc_id, :].data,
                              X_train_tok[doc_id, :].data):
    print(f"{feat:20s} tfidf {weight:.3f}   count {freq}")

15                   tfidf 0.036   count 1
19                   tfidf 0.041   count 1
1976                 tfidf 0.071   count 1
1993                 tfidf 0.036   count 1
1993apr15            tfidf 0.046   count 1
1999                 tfidf 0.351   count 5
22                   tfidf 0.084   count 2
24                   tfidf 0.041   count 1
36                   tfidf 0.052   count 1
action               tfidf 0.052   count 1
after                tfidf 0.030   count 1
again                tfidf 0.037   count 1
airliners            tfidf 0.074   count 1
also                 tfidf 0.026   count 1
and                  tfidf 0.091   count 7
anderson             tfidf 0.070   count 1
anyone               tfidf 0.029   count 1
apr                  tfidf 0.041   count 1
are                  tfidf 0.018   count 1
article              tfidf 0.018   count 1
as                   tfidf 0.096   count 5
astronomy            tfidf 0.054   count 1
at                   tfidf 0.019   count 1
available  

### Which words does IDF push down?

IDF is a property of the corpus, not of a document: one number per term.

In [20]:
import numpy as np

terms = vect.get_feature_names_out()
order = np.argsort(tfidf.idf_)

print("lowest IDF (everywhere, so uninformative):")
print("  ", ", ".join(terms[i] for i in order[:15]))
print("\nhighest IDF (rare, so discriminative):")
print("  ", ", ".join(terms[i] for i in order[-15:]))

lowest IDF (everywhere, so uninformative):
   from, subject, lines, the, organization, to, of, in, and, is, for, it, that, re, on

highest IDF (rare, so discriminative):
   beauty, painful, sarid, 35611, teknikum, pager, digitizing, fasten, 725, luke, dime, luna, dinner, painting, strategies


### Similarity and retrieval

A query is just another document,
vectorized with the **same** vectorizer, and ranked by cosine against the corpus.

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

query = "nasa launched a satellite into orbit around the moon"

q_vec = tfidf.transform(vect.transform([query]))   # same vectorizer, same idf
sims = cosine_similarity(q_vec, X_train_vec).ravel()

top = np.argsort(sims)[::-1][:5]
pd.DataFrame({
    "similarity": sims[top].round(3),
    "category": [train.target_names[y_train[i]] for i in top],
    "snippet": [" ".join(x_train[i].split())[:90] for i in top],
})

,similarity,category,snippet
0,0.368,sci.space,From: thomsonal@cpva.saic.com Subject: Cosmos ...
1,0.330,sci.space,From: henry@zoo.toronto.edu (Henry Spencer) Su...
2,0.272,sci.space,From: HoffmanE@space1.spacenet.jhuapl.edu (Hof...
3,0.267,sci.space,From: C.O.EGALON@LARC.NASA.GOV (CLAUDIO OLIVEI...
4,0.223,sci.space,"From: rls@uihepa.hep.uiuc.edu (Ray Swartz (Oh,..."


The top hits are in the category we expected, and nobody told the model what the
categories were: the ranking comes only from shared words.

### What the vector space cannot do

In [22]:
pairs = [("car", "automobile"), ("car", "cars"), ("moon", "lunar"), ("car", "car")]

for a, b in pairs:
    va = tfidf.transform(vect.transform([a]))
    vb = tfidf.transform(vect.transform([b]))
    print(f"cosine({a:11s}, {b:11s}) = {cosine_similarity(va, vb)[0][0]:.2f}")

cosine(car        , automobile ) = 0.00
cosine(car        , cars       ) = 0.00
cosine(moon       , lunar      ) = 0.00
cosine(car        , car        ) = 1.00


---
# Features that are not words

Bag of words keeps only **which words occur**. Everything else is thrown away: how long the
document is, whether it is written in capitals, how many names it mentions, whether it is
mostly verbs or mostly nouns.

Some of that is signal. Here we build a handful of **linguistic features** with spaCy, put them
on a comparable scale, and attach them to the TF-IDF matrix.

Adapted from the text-processing-to-vectorization lab, which did this on tweets.

In [23]:
# %pip install -q spacy pandas scikit-learn
# !python -m spacy download en_core_web_sm

import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

N_DOCS = 300                     # a sample, to keep the runtime short
sample_texts = [d[:1500] for d in x_train[:N_DOCS]]   # first 1500 characters of each post

docs = list(nlp.pipe(sample_texts, batch_size=32))
print(len(docs), "documents parsed")

300 documents parsed


For each document we count what we care about. Note that these are decisions too: someone
chose *these* features and not others.

In [24]:
rows = []

for raw, doc in zip(sample_texts, docs):
    tokens = [t for t in doc if not t.is_space]
    words = [t for t in tokens if t.is_alpha]
    pos = [t.pos_ for t in tokens]

    rows.append({
        "token_count": len(tokens),
        "type_count": len(set(t.lower_ for t in tokens)),
        "avg_word_len": sum(len(t) for t in words) / max(len(words), 1),
        "verb_count": pos.count("VERB"),
        "noun_count": pos.count("NOUN"),
        "adj_count": pos.count("ADJ"),
        "propn_count": pos.count("PROPN"),
        "ent_count": len(doc.ents),
        "punct_count": pos.count("PUNCT"),
        "upper_ratio": sum(c.isupper() for c in raw) / max(len(raw), 1),
        "digit_ratio": sum(c.isdigit() for c in raw) / max(len(raw), 1),
        "quoted_lines": sum(1 for line in raw.split("\n") if line.startswith(">")),
    })

features_df = pd.DataFrame(rows)
features_df.head()

,token_count,type_count,avg_word_len,verb_count,noun_count,adj_count,propn_count,ent_count,punct_count,upper_ratio,digit_ratio,quoted_lines
0,320,165,4.365217,31,58,18,24,8,57,0.031333,0.013333,20
1,321,160,4.816754,22,69,16,20,14,59,0.038000,0.036000,0
2,250,133,4.994350,20,34,4,56,24,44,0.082172,0.019076,0
3,331,169,4.399061,19,48,19,45,27,76,0.086667,0.030000,12
4,216,139,4.728916,27,35,6,15,9,32,0.034884,0.002683,11


### Inspecting the features

`describe()` tells us whether a feature is constant, nearly always zero, or on a wildly
different scale from the others. Those are the ones to drop or to rescale.

In [25]:
features_df.describe().T

,count,mean,std,min,25%,50%,75%,max
token_count,300.0,223.800000,87.579644,33.000000,142.750000,244.500000,300.000000,364.000000
type_count,300.0,124.923333,39.468002,29.000000,89.000000,132.000000,159.000000,189.000000
avg_word_len,300.0,4.715733,0.382716,3.926230,4.456687,4.676714,4.916620,6.163158
verb_count,300.0,17.506667,9.453457,1.000000,9.750000,17.000000,24.000000,45.000000
noun_count,300.0,34.870000,18.325552,4.000000,21.000000,34.500000,46.000000,185.000000
adj_count,300.0,10.056667,6.600911,0.000000,4.000000,9.000000,15.000000,31.000000
propn_count,300.0,29.696667,13.890566,6.000000,20.000000,27.000000,36.000000,92.000000
ent_count,300.0,16.313333,8.746246,3.000000,9.000000,14.000000,22.000000,55.000000
punct_count,300.0,43.136667,22.191128,6.000000,28.000000,41.000000,52.000000,176.000000
upper_ratio,300.0,0.060081,0.035503,0.014667,0.042667,0.053249,0.068175,0.410000


### Ratios and scaling

Two problems with raw counts:

1. **Length dominates.** A long post has more verbs than a short one, whatever its style.
   Dividing by the token count turns a count into a **proportion**.
2. **Different ranges.** `token_count` runs to the hundreds while `upper_ratio` lives in
   $[0, 1]$. Any model that compares distances, or that follows gradients, will be driven by
   the large-range features alone.

Standardization fixes the second: subtract the mean, divide by the standard deviation, so
every feature has mean 0 and standard deviation 1.

In [26]:
import numpy as np
from sklearn.preprocessing import StandardScaler

f = features_df.copy()

# 1. counts -> proportions of the document
for col in ["verb_count", "noun_count", "adj_count", "propn_count", "ent_count", "punct_count"]:
    f[col.replace("_count", "_ratio")] = f[col] / f["token_count"]

keep = ["token_count", "type_count", "avg_word_len", "upper_ratio", "digit_ratio",
        "quoted_lines", "verb_ratio", "noun_ratio", "adj_ratio", "propn_ratio",
        "ent_ratio", "punct_ratio"]
f = f[keep]

# 2. put every feature on the same scale
scaler = StandardScaler()
F_scaled = scaler.fit_transform(f)

print("feature matrix:", F_scaled.shape)
pd.DataFrame(F_scaled, columns=keep).describe().T[["mean", "std", "min", "max"]].round(2)

feature matrix: (300, 12)


,mean,std,min,max
token_count,-0.0,1.0,-2.18,1.60
type_count,0.0,1.0,-2.43,1.63
avg_word_len,0.0,1.0,-2.07,3.79
upper_ratio,-0.0,1.0,-1.28,9.87
digit_ratio,0.0,1.0,-1.05,8.15
quoted_lines,0.0,1.0,-0.75,3.79
verb_ratio,0.0,1.0,-2.71,2.84
noun_ratio,0.0,1.0,-2.21,8.28
adj_ratio,0.0,1.0,-2.12,2.97
propn_ratio,-0.0,1.0,-1.96,3.80


Nothing about the document changed. Only the units did.

In [27]:
doc_id = 3
pd.DataFrame({"raw": f.iloc[doc_id], "scaled": F_scaled[doc_id].round(2)})

,raw,scaled
token_count,331.000000,1.23
type_count,169.000000,1.12
avg_word_len,4.399061,-0.83
upper_ratio,0.086667,0.75
digit_ratio,0.030000,0.42
quoted_lines,12.000000,1.01
verb_ratio,0.057402,-0.79
noun_ratio,0.145015,-0.25
adj_ratio,0.057402,0.76
propn_ratio,0.135952,-0.16


### Putting the two representations together

The TF-IDF matrix has thousands of sparse columns, one per word. The feature matrix has twelve
dense columns. They describe the same documents, so we can lay them side by side into a single
matrix, and a classifier can use both.

We reuse the TF-IDF rows computed above, for the same documents.

In [28]:
from scipy.sparse import csr_matrix, hstack

X_text = X_train_vec[:N_DOCS]              # TF-IDF, sparse
X_num = csr_matrix(F_scaled)               # linguistic features, made sparse to match

X_combined = hstack([X_text, X_num]).tocsr()

text_names = vect.get_feature_names_out()
combined_names = np.concatenate([text_names, np.array(keep)])

print(f"text features    {X_text.shape[1]}")
print(f"numeric features {X_num.shape[1]}")
print(f"combined         {X_combined.shape}")
print("\nlast 3 text features:", list(text_names[-3:]))
print("first 3 numeric features:", keep[:3])

text features    9468
numeric features 12
combined         (300, 9480)

last 3 text features: ['zuma', 'zumabot', 'zyeh']
first 3 numeric features: ['token_count', 'type_count', 'avg_word_len']


### Which representation is better?

Neither, on its own, is the answer.

- **Bag of words** is high dimensional, sparse and hard to read one coordinate at a time, but
  it carries the content of the document.
- **Linguistic features** are few, dense and interpretable, and they capture *style* rather
  than topic: length, punctuation, how nominal the writing is.

Which one wins depends on the task. Topic labels are mostly in the words; authorship and
readability are mostly in the style. Combining them, as above, is routine.

One caveat for later: scaling matters for models that measure distances or follow gradients.
A decision tree does not care, since it only compares a feature with a threshold.

Every pair of *different* words has similarity 0, including the synonyms. One-hot
coordinates are orthogonal by construction: the representation has no idea that *car* and
*automobile* mean the same thing, or that *car* and *cars* share a lemma.

Stemming or lemmatization fixes the second case. The first one needs a different kind of
representation, which is where embeddings come in later in the course.

---
## Exercises

- Re-run the vectorization with `fetch_20newsgroups(..., remove=("headers", "footers", "quotes"))`.
   How much smaller is the vocabulary? Does the retrieval example still work?
- Add `ngram_range=(1, 2)` to the `CountVectorizer`. How many features now? Look at a
   few bigrams with `get_feature_names_out()`.
- Take a document from the test set as the query and retrieve the most similar training
   documents. Are they in the same category?
- Add a feature of your own to the linguistic features (for example the number of email addresses, or the
   length of the longest word) and check with `describe()` whether it varies across documents.
- Are the linguistic features enough on their own to tell the four categories
   apart? Look at the mean of each feature per category with `groupby`.